# 📚 Public Domain Book Translator — TranslateGemma:27B
### Google Colab · T4 GPU · Run cells **top-to-bottom, one at a time**

| Cell | What it does |
|---|---|
| **1** | Install packages + Ollama + start server (all-in-one) |
| **2** | Download `translategemma:27b` (~17 GB, cached after first run) |
| **3** | Configure translation settings |
| **4** | Upload your `.txt` book file |
| **5** | Run translation |
| **6** | Download translated file |

In [ ]:
# ════════════════════════════════════════════════════
# CELL 1 — Install packages + Ollama + start server
# ════════════════════════════════════════════════════
import subprocess, time, os

# 1a. Install zstd (needed by Ollama installer)
print('📦 Installing system dependencies...')
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1

# 1b. Install Ollama binary
print('🦙 Installing Ollama...')
!curl -fsSL https://ollama.com/install.sh | sh

# 1c. Install Python packages
print('\n📦 Installing Python packages...')
!pip install -q ollama ipywidgets

# 1d. Set host env var BEFORE starting server
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'

# 1e. Start Ollama server in background
print('\n🚀 Starting Ollama server in background...')
subprocess.Popen(
    ['/usr/local/bin/ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 1f. Wait then verify
time.sleep(5)
try:
    import ollama
    ollama.list()
    print('✅ Ollama server is running!')
except Exception as e:
    print(f'⚠️  Server not ready yet: {e}')
    print('   Wait 10 seconds and re-run this cell once.')


In [ ]:
# ════════════════════════════════════════════════════
# CELL 2 — Download translategemma:27b
# ⚠  ~17 GB — first run takes 15–30 min
# ⚠  Cached for the rest of the Colab session
# ════════════════════════════════════════════════════
import ollama

MODEL = 'translategemma:27b'

# Check if already downloaded
_already = False
try:
    _models = ollama.list().get('models', [])
    _names  = [m.get('model', '') or m.get('name', '') for m in _models]
    _already = any(MODEL in n for n in _names)
except Exception:
    pass

if _already:
    print(f'✅ {MODEL} is already downloaded — skipping pull.')
else:
    print(f'📥 Pulling model: {MODEL}')
    print('   This will take 15–30 minutes on first run...\n')

    _current_digest = ''
    try:
        for _progress in ollama.pull(MODEL, stream=True):
            _digest = _progress.get('digest', '')
            if _digest != _current_digest and _current_digest:
                print()  # newline between layers
            _current_digest = _digest

            _status = _progress.get('status', '')
            # Only do math when BOTH keys exist AND are not None
            if 'completed' in _progress and 'total' in _progress \
                    and _progress['completed'] is not None \
                    and _progress['total'] is not None \
                    and _progress['total'] > 0:
                _done  = _progress['completed']
                _total = _progress['total']
                _pct   = _done / _total * 100
                print(f'\r   {_status}: {_pct:.1f}% ({_done}/{_total})',
                      end='', flush=True)
            else:
                print(f'\r   {_status}', end='', flush=True)

        print(f'\n\n✅ Model pulled successfully!')

    except Exception as _e:
        print(f'\n❌ Pull error: {_e}')
        print('   Make sure Cell 1 ran successfully before running this cell.')
        raise

# Final model list
print('\n📋 Available Ollama models:')
try:
    for _m in ollama.list().get('models', []):
        _n = _m.get('model', '') or _m.get('name', 'unknown')
        _s = _m.get('size', 0) / 1024**3
        print(f'   • {_n} ({_s:.2f} GB)')
except Exception as _le:
    print(f'   (Could not list models: {_le})')


In [ ]:
# ════════════════════════════════════════════════════
# CELL 3 — Configure translation settings
# ════════════════════════════════════════════════════
import ipywidgets as widgets
from IPython.display import display

_W  = dict(style={'description_width': '160px'},
           layout=widgets.Layout(width='500px'))
_SW = dict(style={'description_width': '160px'},
           continuous_update=False,
           layout=widgets.Layout(width='540px'))

language_input = widgets.Text(
    value='Japanese',
    placeholder='e.g. Japanese, Korean, French, Spanish, Arabic, Russian…',
    description='Source Language:',
    **_W)

title_input = widgets.Text(
    value='',
    placeholder='Optional — keeps proper nouns consistent',
    description='Book Title:',
    **_W)

author_input = widgets.Text(
    value='',
    placeholder='Optional — helps match author style',
    description='Author:',
    **_W)

_genres = [
    'Literary Fiction','Classic Literature','Historical Fiction',
    'Romance','Romantic Drama','Mystery','Detective Fiction',
    'Thriller','Psychological Thriller','Horror','Gothic Fiction',
    'Science Fiction','Fantasy','Mythological Fantasy',
    'Adventure','Action & Adventure','Satire','Dark Satire',
    'Tragedy','Comedy','Tragicomedy','Drama','Social Drama',
    'Philosophical Fiction','Existential Fiction','Political Fiction',
    'Magical Realism','Surrealism','Mythology & Folklore',
    'Fable & Allegory','Short Story Collection','Novella',
    'Epic Poetry','Lyric Poetry','Narrative Poetry',
    'Biography','Autobiography','Memoir','Personal Essays',
    'Travel Writing','War Fiction','Samurai / Period Drama',
    'Wuxia / Martial Arts Fiction','Court Drama',
    'Coming-of-Age / Bildungsroman','Epistolary Fiction','Other'
]
genre_dropdown = widgets.Dropdown(
    options=_genres, value='Literary Fiction',
    description='Genre:', **_W)

_narr = [
    'First-person / Past tense',
    'First-person / Present tense',
    'First-person / Mixed tense',
    'Third-person Limited / Past tense',
    'Third-person Limited / Present tense',
    'Third-person Omniscient / Past tense',
    'Third-person Omniscient / Present tense',
    'Second-person / Past tense',
    'Second-person / Present tense',
    'Multiple POV / Past tense',
    'Multiple POV / Present tense',
    'Epistolary (Letters and Diary entries)',
    'Stream of Consciousness',
    'Unreliable Narrator',
    'Frame Narrative (story within a story)',
    'Collective / We-narrator',
    'Non-linear / Fragmented',
]
narrative_dropdown = widgets.Dropdown(
    options=_narr, value='Third-person Omniscient / Past tense',
    description='Narrative Style:', **_W)

chunk_slider = widgets.IntSlider(
    value=1500, min=300, max=4000, step=100,
    description='Chunk Size (chars):', **_SW)

overlap_slider = widgets.IntSlider(
    value=250, min=0, max=800, step=50,
    description='Overlap (chars):', **_SW)

# All labels inside VBox must be widgets.HTML (NOT IPython.display.HTML)
display(widgets.VBox(
    children=[
        widgets.HTML('<h3 style="margin:4px 0">📖 Translation Settings</h3>'),
        widgets.HTML('<b>Source Language</b>'),
        language_input,
        widgets.HTML('<b>Metadata (optional — improves proper-noun consistency)</b>'),
        title_input,
        author_input,
        widgets.HTML('<b>Style</b>'),
        genre_dropdown,
        narrative_dropdown,
        widgets.HTML('<b>Chunking</b>'),
        chunk_slider,
        widgets.HTML('<span style="color:gray;font-size:0.85em">'
                     'Larger = better coherence, slower. Recommended: 1000-2000</span>'),
        overlap_slider,
        widgets.HTML('<span style="color:gray;font-size:0.85em">'
                     'Tail of previous translation fed as context. Recommended: 150-350</span>'),
        widgets.HTML('<p style="color:green;margin-top:10px">'
                     '<b>All set — run Cell 4 to upload your file.</b></p>'),
    ],
    layout=widgets.Layout(grid_gap='6px')
))


In [ ]:
# ════════════════════════════════════════════════════
# CELL 4 — Upload your .txt book file
# ════════════════════════════════════════════════════
from google.colab import files

print("📂 Click 'Choose Files' and select your .txt file:")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError('No file uploaded. Re-run this cell and select a file.')

_fname = list(uploaded.keys())[0]
_raw   = uploaded[_fname]

# Auto-detect encoding — try most common ones in order
content = None
_detected_enc = 'unknown'
for _enc in ('utf-8-sig', 'utf-8', 'shift_jis', 'euc-kr',
             'gb2312', 'big5', 'latin-1', 'cp1252'):
    try:
        content = _raw.decode(_enc)
        _detected_enc = _enc
        break
    except (UnicodeDecodeError, LookupError):
        continue

if content is None:
    content = _raw.decode('latin-1', errors='replace')
    _detected_enc = 'latin-1 (fallback)'

print(f'\n✅ File     : {_fname}')
print(f'   Encoding : {_detected_enc}')
print(f'   Chars    : {len(content):,}')
print(f'   Words    : {len(content.split()):,}')
print(f'   ~Chunks  : ~{len(content) // chunk_slider.value + 1} '
      f'(at {chunk_slider.value} chars/chunk)')
print('\n✅ File ready — run Cell 5 to start translation.')


In [ ]:
# ════════════════════════════════════════════════════
# CELL 5 — Run translation
# ════════════════════════════════════════════════════
import ollama, time

# Collect widget values
source_language = language_input.value.strip() or 'Unknown'
book_title      = title_input.value.strip()
book_author     = author_input.value.strip()
genre           = genre_dropdown.value
narrative_style = narrative_dropdown.value
chunk_size      = chunk_slider.value
overlap_size    = overlap_slider.value

# System prompt
SYSTEM_PROMPT = (
    'You are an expert literary translator specializing in translating '
    'public domain books into natural, modern English. '
    'Your goal is meaning-for-meaning translation — preserving the '
    "author's intent, emotional tone, narrative flow, and cultural "
    'nuance while making it feel as if the book was originally written '
    'in contemporary English.\n\n'
    'PRINCIPLES:\n'
    '1. MODERN NATURAL ENGLISH\n'
    '   - Use natural, fluent, conversational modern English.\n'
    '   - Avoid archaic phrasing unless the source uses it intentionally.\n'
    '   - Replace outdated idioms with modern equivalents of the same emotional weight.\n\n'
    '2. CONTEXT AND VOICE PRESERVATION\n'
    '   - Never treat any sentence in isolation — carry forward all prior context.\n'
    '   - Preserve character voice: formal, casual, humorous, gruff — keep it consistent.\n'
    '   - Preserve the exact narrative tense and point of view from the source.\n\n'
    '3. CULTURAL NUANCE\n'
    '   - Use the closest modern English approximation for untranslatable concepts.\n'
    '   - Add a brief inline note ONLY if essential — e.g., [a traditional rice wine].\n'
    '   - Do not over-explain. Trust the reader.\n\n'
    '4. DIALOGUE\n'
    '   - Dialogue must sound like real people speaking today, not theatrical.\n'
    '   - Preserve subtext, hesitation, emotion, and personality in every line.\n\n'
    '5. LITERARY DEVICES\n'
    '   - Preserve metaphors, similes, symbolism, and imagery as closely as possible.\n'
    '   - If a metaphor cannot translate directly, use an equivalent that evokes the same feeling.\n\n'
    '6. FORMATTING\n'
    '   - Maintain original paragraph breaks, chapter headings, and structure.\n'
    '   - Do not merge or split paragraphs unless grammatically unavoidable.\n\n'
    'CRITICAL OUTPUT RULE: Output ONLY the translated English text. '
    'No commentary, no translator notes, no meta-text, no preamble, no sign-off. '
    'Just the translation itself.'
)

# Smart chunker — splits at paragraph/sentence boundaries
def split_into_chunks(text, size):
    chunks, start = [], 0
    n = len(text)
    while start < n:
        end = min(start + size, n)
        if end < n:
            pb = text.rfind('\n\n', start, end)
            if pb != -1 and pb > start + size // 3:
                end = pb + 2
            else:
                best = -1
                for sep in ['.\n', '!\n', '?\n', '\n',
                             '. ', '! ', '? ', '。', '！', '？']:
                    idx = text.rfind(sep, start + size // 3, end)
                    if idx > best:
                        best = idx
                if best != -1:
                    end = best + 1
        chunks.append(text[start:end])
        start = end
    return chunks

chunks = split_into_chunks(content, chunk_size)
total  = len(chunks)

print('=' * 58)
print(f'🌐 Language  : {source_language} -> English')
print(f'📚 Genre     : {genre}')
print(f'📝 Style     : {narrative_style}')
if book_title:  print(f'📖 Title     : {book_title}')
if book_author: print(f'✍  Author    : {book_author}')
print(f'🔢 Chunks    : {total}  ({chunk_size} chars, {overlap_size} overlap)')
print('=' * 58)

translated_chunks = []
prev_tail         = ''
start_time        = time.time()

for i, chunk in enumerate(chunks):
    t0 = time.time()

    meta = ''
    if book_title:  meta += f'Book title: {book_title}\n'
    if book_author: meta += f'Author: {book_author}\n'

    ctx = (
        'Previously translated passage '
        '(for continuity only — do NOT include this in your output):\n'
        + prev_tail + '\n\n'
    ) if prev_tail else 'This is the very beginning of the text.\n\n'

    user_prompt = (
        f'Translate the following passage from {source_language} to English.\n\n'
        f'{meta}'
        f'Genre: {genre}\n'
        f'Narrative style: {narrative_style}\n\n'
        f'{ctx}'
        'Translate ONLY the passage below. '
        'Output the English translation and nothing else:\n'
        '---\n'
        f'{chunk}\n'
        '---'
    )

    # FIX B: Retry logic — up to 3 attempts per chunk
    translation = f'[TRANSLATION ERROR - CHUNK {i+1}]'
    for _attempt in range(3):
        try:
            resp = ollama.chat(
                model='translategemma:27b',
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': user_prompt},
                ],
                options={'temperature': 0.25, 'top_p': 0.92,
                         'num_predict': 1200,
                         'num_ctx': 4096}
            )
            _raw = resp['message']['content'].strip()
            # FIX C: Basic validation — reject obvious garbage
            _words = _raw.split()
            _src_words = chunk.split()
            _ratio = len(_words) / max(len(_src_words), 1)
            _has_error_marker = '[TRANSLATION' in _raw or 'ERROR' in _raw[:30]
            # Accept if: non-empty, ratio 0.3-3.0, no error marker
            if _raw and 0.3 <= _ratio <= 3.0 and not _has_error_marker:
                translation = _raw
                break  # good output — stop retrying
            else:
                print(f'  ⚠ Chunk {i+1} attempt {_attempt+1}: '
                      f'rejected (ratio={_ratio:.2f}, empty={not _raw}) — retrying...')
        except Exception as err:
            print(f'  ⚠ Chunk {i+1} attempt {_attempt+1} error: {err}')
    if translation.startswith('[TRANSLATION ERROR'):
        print(f'  ❌ Chunk {i+1}: all 3 attempts failed — placeholder inserted')

    translated_chunks.append(translation)
    if overlap_size > 0:
        prev_tail = translation[-overlap_size:]

    elapsed       = time.time() - t0
    total_elapsed = time.time() - start_time
    eta_min       = (total_elapsed / (i + 1)) * (total - i - 1) / 60
    print(f'  Done [{i+1:>3}/{total}]  '
          f'{len(translation):>5} chars | {elapsed:>5.1f}s | ETA {eta_min:.1f} min')

print('\n' + '=' * 58)
print(f'Done!  Total time: {(time.time()-start_time)/60:.1f} min')
final_translation = '\n\n'.join(translated_chunks)
print(f'Final output: {len(final_translation):,} chars')
print('Run Cell 6 to download your file.')


In [ ]:
# ════════════════════════════════════════════════════
# CELL 6 — Save and download translated file
# ════════════════════════════════════════════════════
from google.colab import files
import os, re

_safe    = re.sub(r'[^\w\-.]', '_', _fname.replace('.txt', ''))
_outpath = f'/content/{_safe}_EN_translated.txt'

with open(_outpath, 'w', encoding='utf-8') as _f:
    _header = ['TRANSLATED TO ENGLISH']
    if book_title:  _header.append(f'Title  : {book_title}')
    if book_author: _header.append(f'Author : {book_author}')
    _header += [
        f'From   : {source_language}',
        f'Genre  : {genre}',
        f'Style  : {narrative_style}',
        '=' * 58, ''
    ]
    _f.write('\n'.join(_header) + '\n')
    _f.write(final_translation)

_kb = os.path.getsize(_outpath) / 1024
print(f'Saved  : {_outpath}  ({_kb:.1f} KB)')
print('Downloading to your computer...')
files.download(_outpath)
print("Done! Check your browser's Downloads folder.")
